In [ ]:
#This is pyenv file for making AI classification.

In [21]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()
import time

In [22]:
data=pd.read_csv('data2005.csv')

In [23]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [24]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
4590,79807,09V246000,TRIPLE E RV,COMMANDER,2005,50901-C,"SERVICE BRAKES, HYDRAULIC",TRIPLE E RECREATIONAL VEHICLES,NaN,NaN,...,TRIPLE E IS RECALLING 16 MOTORHOMES EQUIPPED W...,THE DRIVER WOULD NORMALLY DETECT THE BRAKE DRA...,DEALERS WILL INSPECT AND REPLACE THE NECESSARY...,TRIPLE E RECALL NO. 50901-C.OWNERS MAY ALSO CO...,32078000215213000000062,NaN,NaN,NaN,No,No
1105,130910,17V328000,BMW,760I,2005,NaN,LATCHES/LOCKS/LINKAGES:DOORS:LATCH,"BMW of North America, LLC",20040823.0,20080724.0,...,"BMW of North America, LLC (BMW) is recalling c...",The door may unexpectedly open due to road or ...,"BMW will notify owners, and dealers will updat...",Owners may also contact the National Highway T...,70306000204672000000289,NaN,NaN,NaN,No,No
1198,139753,19V828000,SPARTAN,GLADIATOR,2005,19V-674,"SERVICE BRAKES, AIR:SUPPLY:RESERVOIR/TANK",Marion Body Works Inc.,20040101.0,20191231.0,...,Marion Body Works Inc. (Marion Body Works) is ...,Without the control line being pressure-protec...,"Marion Body Works will notify owners, and deal...",Owners may also contact the National Highway T...,100706000838933000000095,NaN,NaN,NaN,No,No
878,117814,16V501000,NEWMAR,DUTCH STAR,2005,16V 313,ENGINE AND ENGINE COOLING:EXHAUST SYSTEM,Newmar Corporation,19971212.0,20150527.0,...,Newmar Corporation (Newmar) is recalling certa...,If hot exhaust gas contacts other components o...,"Newmar will notify owners, and Spartan dealers...",Owners may also contact the National Highway T...,59157000114157000000138,NaN,NaN,NaN,No,No
3959,72298,08V320000,FLEETWOOD,EXCURSION,2005,NaN,EQUIPMENT:RECREATIONAL VEHICLE/TRAILER,"FLEETWOOD ENTERPRISES, INC.",20030401.0,20061130.0,...,"FLEETWOOD IS RECALLING 70,101 MY 2003-2007 SOU...",IF THIS SOLUTION IS EXPOSED TO AN IGNITION SOU...,FLEETWOOD IS WORKING WITH DOMETIC TO REPAIR TH...,CUSTOMERS MAY ALSO CONTACT THE NATIONAL HIGHWA...,27396000207813000000330,NaN,NaN,NaN,No,No


In [25]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

In [26]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [10]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [27]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-nano):  gpt-5.4-mini


In [28]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [29]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $0.71'

In [30]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

def classify_fix(prompt_including_tweet):
    for attempt in range(3):
        try:
            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result
        except Exception as e:
            time.sleep(attempt*3)
    return None    

In [31]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify_fix)

100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:59<00:00,  1.19s/it]


In [17]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
4590,TRIPLE E IS RECALLING 16 MOTORHOMES EQUIPPED WITH AXLES THAT INCLUDE BOSCH ZOPS OR ZOHT PIN SLIDE HYDRAULIC DISC BRAKES. WHEN COMBINED WITH OCCASIONAL OR SEASONAL VEHICLE OPERATION MAY EXPERIENCE CALIPERS STICKING IN THE APPLIED POSITION. THIS CAN RESULT IN ABNORMAL HEAT GENERATION AT THE WHEEL END CAUSING BRAKE DRAG.,safety_equipment
1105,"BMW of North America, LLC (BMW) is recalling certain 2005-2008 745i, 745Li, 750i, 750Li, 760i, 760Li, and B7 Alpina vehicles equipped with both the Comfort Access and Soft Close Automatic options. In these vehicles, the doors may appear to be closed and latched, but, in fact, may inadvertently open.",safety_equipment
1198,Marion Body Works Inc. (Marion Body Works) is recalling certain 2005-2020 Emergency Response Gladiator and Metrostar vehicles equipped with pneumatically-controlled automatic tire chain systems. The air pressure control lines may not be pressure-protected.,others
878,"Newmar Corporation (Newmar) is recalling certain model year 1998-2010 Dutch Star, 2008 Kountry Star, 2008-2009 All Star and Ventana, 2009-2010 Dutch Aire, 2014-2015 King Aire and 2015 Essex motorhomes, built on a Spartan Motors chassis. In the affected vehicles, the flexible exhaust tube may tear allowing hot exhaust to blow against other components or materials.",others
3959,"FLEETWOOD IS RECALLING 70,101 MY 2003-2007 SOUTHWIND, PACE ARROW, BOUNDER, FLAIR, STORM, FIESTA, TERRA GAS CLASS A MOTOR HOMES; BOUNDER, DISCOVERY, PROVIDENCE, EXCURSION, EXPEDITION, REVOLUTION, AMERICAN EAGLE, AMERICAN HERITAGE, AMERICAN TRADITION DIESEL CLASS A MOTOR HOMES; JAMBOREE TIOGA CLASS C MOTOR HOMES; AND PROWLER, WILDERNESS, TERRY, MALLARD, PIONEER, PRIDE AND TRIUMPH CONVENTIONAL AND FIFTH WHEEL TRAVEL TRAILERS EQUIPPED WITH A TWO-DOOR REFRIGERATORS MANUFACTURED BY THE DOMETIC CO...",others
256,"TOYOTA IS RECALLING CERTAIN MODEL YEAR 2004 AVALON VEHICLES; 2004 AND 2005 CAMRY, CAMRY SOLARA, HIGHLANDER, AND SIENNA VEHICLES, 2006 HIGHLANDER HYBRID; 2004 AND 2005 LEXUS ES330 AND RX330; AND 2006 LEXUS RX400H VEHICLES MANUFACTURED FROM JUNE 1, 2004 THROUGH MARCH 31, 2005, EQUIPPED WITH THE 1MZ-FE OR 3MZ-FE ENGINE. THE AMOUNT OF ADHESIVE AGENT APPLIED BETWEEN THE OUTER RING AND THE TORSIONAL RUBBER DAMPER (INNER RING) IN THE CRANKSHAFT PULLEY MAY BE INADEQUATE. IF THE ADHESIVE IS INSUFFI...",powertrain
3056,"CERTAIN TRAILERS FAIL TO COMPLY WITH THE REQUIREMENTS OF FEDERAL MOTOR VEHICLE SAFETY STANDARD NO. 223, ""REAR IMPACT GUARDS."" PROPER REAR IMPACT GUARDS WERE NOT INSTALLED ON THESE TRAILERS.",safety_equipment
3485,"CERTAIN MY 2005 AND 2006 U.S. BUS UNIVERSE AND STURDIBUS SCHOOL BUSES MANUFACTURED BETWEEN SEPTEMBER 1, 2005, AND SEPTEMBER 1, 2006. THESE BUSES DO NOT HAVE THE TIRE AND LOADING DECAL WHICH DOES NOT CONFORM TO FEDERAL MOTOR VEHICLE SAFETY STANDARD NO. 110, TIRE SELECTION AND RIMS.",safety_equipment
2994,"ON CERTAIN VANS EQUIPPED WITH THE POWER LIFTGATE OPTION, THE POWER LIFTGATE MOTOR AUTOMATICALLY DISENGAGES AFTER THE LIFTGATE IS POWERED OPEN.",others
1140,"Daimler Trucks North America LLC (DTNA) is recalling certain 2002-2017 Thomas Built Buses Saf-T-Liner C2, Saf-T-Liner HDX, FS65, and Minotour school buses, equipped with Kidde Plastic-Handle or Push Button 'Pindicator' Fire Extinguishers. These extinguishers may become clogged, preventing the extinguisher from discharging as expected or requiring excessive force to activate the extinguisher. Additionally, in certain models, the nozzle may detach from the valve assembly with enough force th...",safety_equipment


In [32]:
okay_to_do_the_whole_thing = input()

In [33]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [35]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify_fix)
data.to_csv(data_dir + 'Recall_classified2005.csv')

100%|██████████████████████████████████████████████████████████████████████████████| 4673/4673 [1:41:10<00:00,  1.30s/it]


In [36]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-fa3...4da",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 1.6227260000000001,
    "include_byok_in_limit": false,
    "usage": 45.47633575,
    "usage_daily": 18.377274,
    "usage_weekly": 39.8178143,
    "usage_monthly": 43.54938185,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}
